In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.ao.quantization as quant
from torch.ao.quantization import QuantStub, DeQuantStub
import torch.nn.functional as F
import numpy as np

# =====================================================================
# 0. PLANTA DA REDE
# =====================================================================
class LeNet5_Quantized(nn.Module):
    def __init__(self, activation_type="relu"):
        super(LeNet5_Quantized, self).__init__()
        self.quant = QuantStub()
        self.dequant = DeQuantStub()
        self.activation = nn.ReLU()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.quant(x)
        x = self.pool1(self.activation(self.conv1(x)))
        x = self.pool2(self.activation(self.conv2(x)))
        x = x.reshape(-1, 16 * 5 * 5)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        x = self.dequant(x)
        return x

# =====================================================================
# 1. CARREGANDO DADOS E MODELO FLOAT
# =====================================================================
print("=== 1. PREPARANDO AMBIENTE ===")
transform = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

model = LeNet5_Quantized(activation_type="relu")
model.load_state_dict(torch.load("lenet5_float.pth"))
model.eval()
print("✓ Modelo Float carregado.")

# =====================================================================
# 2. QUANTIZAÇÃO SIMÉTRICA (PERFEITA PARA FPGA)
# =====================================================================
print("\n=== 2. QUANTIZANDO PARA O HARDWARE ===")
act_obs = quant.MovingAverageMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_tensor_symmetric)
weight_obs = quant.MovingAveragePerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric)
model.qconfig = quant.QConfig(activation=act_obs, weight=weight_obs)

model_prepared = quant.prepare(model)

print("Calibrando escalas...")
with torch.no_grad():
    for i, (inputs, _) in enumerate(train_loader):
        model_prepared(inputs)
        if i > 50: break 

model_int8 = quant.convert(model_prepared)
torch.save(model_int8.state_dict(), "lenet5_quantized_simetrica.pth")
print("✓ Novo modelo simétrico salvo!")

=== 1. PREPARANDO AMBIENTE ===
✓ Modelo Float carregado.

=== 2. QUANTIZANDO PARA O HARDWARE ===
Calibrando escalas...


C:\Users\Norian\AppData\Local\Temp\ipykernel_26064\3179537353.py:59: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = quant.prepare(model)


✓ Novo modelo simétrico salvo!

=== 3. CÁLCULO DO SHIFT ===
=> ATUALIZE SEU VIVADO: SHIFT := 11; <=

=== 4. GABARITO DA CONV 1 (EMULANDO VHDL) ===

=== 5. EXPORTAÇÃO DE ARQUIVOS ===
✓ 'input_image.txt' atualizado!
✓ 'conv1_params.txt' atualizado!

Fluxo finalizado! O PyTorch agora está rodando uma simulação perfeita do seu circuito.


C:\Users\Norian\AppData\Local\Temp\ipykernel_26064\3179537353.py:67: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = quant.convert(model_prepared)


In [ ]:
import numpy as np
import torch

print("\n=== CÁLCULO DOS SHIFTS (C1 E C2) ===")

# --- SHIFT DA C1 ---
# A entrada da C1 é a imagem original quantizada (model_int8.quant)
s_in_c1 = model_int8.quant.scale
if hasattr(s_in_c1, 'item'): s_in_c1 = s_in_c1.item()

s_weight_c1 = model_int8.conv1.weight().q_per_channel_scales()[0]
if hasattr(s_weight_c1, 'item'): s_weight_c1 = s_weight_c1.item()

s_out_c1 = model_int8.conv1.scale
if hasattr(s_out_c1, 'item'): s_out_c1 = s_out_c1.item()

M_c1 = (s_in_c1 * s_weight_c1) / s_out_c1
real_shift_c1 = int(np.round(-np.log2(M_c1)))

print(f"=> ATUALIZE SEU VIVADO (CAMADA C1): SHIFT := {real_shift_c1}; <=")

# --- SHIFT DA C2 ---
# A entrada da C2 é a saída da C1!
s_in_c2 = model_int8.conv1.scale  
if hasattr(s_in_c2, 'item'): s_in_c2 = s_in_c2.item()

s_weight_c2 = model_int8.conv2.weight().q_per_channel_scales()[0]
if hasattr(s_weight_c2, 'item'): s_weight_c2 = s_weight_c2.item()

s_out_c2 = model_int8.conv2.scale
if hasattr(s_out_c2, 'item'): s_out_c2 = s_out_c2.item()

M_c2 = (s_in_c2 * s_weight_c2) / s_out_c2
real_shift_c2 = int(np.round(-np.log2(M_c2)))

print(f"=> ATUALIZE SEU VIVADO (CAMADA C2): SHIFT := {real_shift_c2}; <=")

print("\n=== EXPORTAÇÃO DOS ARQUIVOS .TXT ===")

def export_conv_params_only(layer_name, layer_obj):
    """
    Exporta APENAS parâmetros de camadas Convolucionais (4D)
    Alinhado para BRAMs de 128 bits (16 bytes por linha).
    """
    all_rows = []

    # ==========================================
    # 1. PROCESSAMENTO DOS BIAS
    # ==========================================
    # A LeNet-5 tem poucos bias nas Convs (C1 = 6, C2 = 16)
    bias_float = layer_obj.bias().detach().numpy()
    scale = layer_obj.scale
    if hasattr(scale, 'item'): scale = scale.item()
    bias_int8 = np.round(bias_float / scale).astype(np.int8)
    
    # Empacota os Bias de 16 em 16 bytes
    for i in range(0, len(bias_int8), 16):
        chunk = bias_int8[i:i+16]
        padded_chunk = np.zeros(16, dtype=np.int8)
        padded_chunk[:len(chunk)] = chunk
        all_rows.append(padded_chunk)

    # ==========================================
    # 2. PROCESSAMENTO DOS PESOS (WEIGHTS)
    # ==========================================
    # O shape aqui SEMPRE será: [Filtros, Canais, Altura, Largura]
    weights_nd = layer_obj.weight().int_repr().numpy()
    num_filters = weights_nd.shape[0]
    num_channels = weights_nd.shape[1]

    for f in range(num_filters):
        for c in range(num_channels):
            # Isola um kernel 5x5 exato (25 bytes)
            kernel_5x5 = weights_nd[f, c].flatten()
            
            # Fatiador: Pega o kernel de 25 bytes e divide em linhas de 16
            # Vai gerar sempre 2 linhas por kernel:
            # Linha 1: 16 bytes cheios
            # Linha 2: 9 bytes úteis + 7 bytes de "zeros" (Padding)
            for i in range(0, len(kernel_5x5), 16):
                chunk = kernel_5x5[i:i+16]
                padded_chunk = np.zeros(16, dtype=np.int8)
                padded_chunk[:len(chunk)] = chunk
                all_rows.append(padded_chunk)

    # ==========================================
    # 3. GERAÇÃO DO ARQUIVO .TXT
    # ==========================================
    filename = f"{layer_name}_params_aligned.txt"
    with open(filename, "w") as file:
        for row in all_rows:
            # Hexadecimal invertido (Little Endian)
            hex_str = "".join(f"{b & 0xFF:02X}" for b in reversed(row))
            file.write(f"0x{hex_str}\n")
            
    print(f"✓ '{filename}' gerado! (Estrutura: {num_filters} Filtros x {num_channels} Canais)")

# =====================================================================
# EXECUÇÃO APENAS PARA C1 E C2
# =====================================================================
print("Exportando camadas Convolucionais...")
export_conv_params_only("conv1", model_int8.conv1) 
export_conv_params_only("conv2", model_int8.conv2)
print("Pronto para enviar para o VHDL!")


=== CÁLCULO DO SHIFT E EXPORTAÇÃO DA C2 ===
=> ATUALIZE SEU VIVADO (CAMADA C2): SHIFT := 11; <=
Exportando camadas Convolucionais...
✓ 'conv1_params_aligned.txt' gerado! (Estrutura: 6 Filtros x 1 Canais)
✓ 'conv2_params_aligned.txt' gerado! (Estrutura: 16 Filtros x 6 Canais)
Pronto para enviar para o VHDL!
